# GENE7 morphology — UMAP views

`gene7_morphospace.ipynb` works in the linear PCA basis. This asks what a nonlinear embedding adds,
and whether it should run on the **full** 80-dimensional biological latent block (`z_mu_b_*`) or on a
**denoised** reduction (the first 10 GENE7-native PCs). 2D and 3D layouts for both input spaces.

**Requires `vae-env-cluster`** — it is the environment with both `umap-learn` and the `openpyxl`
that `build_master_table` needs for the plate hash maps. `morphseq-env` has umap but not openpyxl.

### How to read a UMAP

Only the neighbourhood structure is meaningful. Distances between separated blobs are not, the axes
have no units, and the layout moves with `n_neighbors`. So every layout below is reported with:

- **trustworthiness** — fraction of each point's embedded neighbours that were also neighbours in the
  input space (local fidelity, the thing UMAP optimises),
- **Spearman rho** on pairwise distances (global fidelity, which UMAP does *not* promise), and
- a **kNN label purity** table against a chance baseline and a label shuffle, so "it looks
  organised by temperature" becomes a number.

There is also an `n_neighbors` sweep, because at n = 553 the layout is genuinely sensitive to it.

In [ ]:
import sys
import warnings
from pathlib import Path

HERE = Path.cwd()
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(HERE.parents[2] / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.io as pio
from IPython.display import display

import plotting as pl
import gene7_umap as gu
import gene7_config as cfg
import morph_pca_spline as mps
from morphseq_integration import build_master_table

# umap-learn emits a numba n_jobs warning whenever random_state is set (single-threaded by design).
warnings.filterwarnings("ignore", category=UserWarning, module="umap")

pio.renderers.default = "notebook"
FIGS = cfg.figure_dir("umap")
print(f"figures -> {FIGS}")

# Same loader the rest of the GENE7 work uses (see run_cohort_axes.py), so the well set is identical.
gene7 = mps.gene7_latents_from_master(build_master_table().query("has_morph").copy())
gene7 = pl.add_perturbation_column(gene7).reset_index(drop=True)

META_COLUMNS = ["well_id", "target", "perturbation_group", "temperature", "stage", "experiment_id"]
metadata = gene7[META_COLUMNS]

print(f"{len(gene7)} wells | {len(gu.latent_columns(gene7))} biological latent dims")
print(f"  temperatures {sorted(gene7.temperature.unique())}")
print(f"  stages       {sorted(gene7.stage.unique())} hpf")
print(f"  groups       {sorted(gene7.perturbation_group.unique())}")

## The two input spaces

The question "full space or reduced space?" has a prerequisite: how many dimensions does this latent
block actually use? If the answer is "about five", then the 80D and 10-PC inputs are the same input
and the comparison is a formality rather than a choice.

In [ ]:
from sklearn.decomposition import PCA

_latents = gene7.loc[:, gu.latent_columns(gene7)].to_numpy(dtype=float)
_spectrum = PCA(random_state=gu.RANDOM_STATE).fit(_latents)
_cumulative = np.cumsum(_spectrum.explained_variance_ratio_)

spectrum = pd.DataFrame({
    "component": np.arange(1, 11),
    "explained_variance_ratio": _spectrum.explained_variance_ratio_[:10],
    "cumulative": _cumulative[:10],
})
display(spectrum.round(4))

for threshold in (0.90, 0.95, 0.99, 0.999):
    print(f"  {threshold:.1%} of latent variance needs "
          f"{int(np.searchsorted(_cumulative, threshold) + 1):2d} of {_latents.shape[1]} PCs")

spaces = gu.build_spaces(gene7)
print()
for space in spaces.values():
    print(f"  {space.name:5} {str(space.matrix.shape):11} {space.note}")

## Fit the layouts

Four embeddings: {full, first 10 PCs} x {2D, 3D}, all at `n_neighbors=15`, `min_dist=0.1`,
`random_state=42`. Nothing is standardised before embedding — the biological latents share a common
prior scale and the PC basis is unwhitened, so a per-feature rescale would re-inflate exactly the
degenerate tail the reduction exists to drop.

In [ ]:
embeddings = gu.run_embeddings(spaces, dimensions=(2, 3))

quality = pd.DataFrame(
    [{"space": spaces[key].label, "dims": f"{n}D", **result.quality}
     for (key, n), result in embeddings.items()]
).set_index(["space", "dims"])
display(quality.round(3))

## 2D layouts

Rows are the input space, columns are what the colour encodes.

In [ ]:
figure = gu.grid_2d(embeddings, metadata, spaces)
figure.savefig(FIGS / "umap_2d_grid.png", dpi=200, bbox_inches="tight")
figure.savefig(FIGS / "umap_2d_grid.pdf", bbox_inches="tight")
plt.show()
print("wrote umap_2d_grid.{png,pdf}")

## 3D layouts

Exported as interactive HTML rather than static images: a fixed camera on a 3D cloud is one
arbitrary projection, and the rotation is the part that carries the information.

In [ ]:
for key in ("full", "pc10"):
    frame_3d = embeddings[(key, 3)].to_frame(metadata)
    figure_3d = gu.umap_3d_figure(
        frame_3d,
        color_by="temperature",
        title=f"GENE7 morphology UMAP 3D — {spaces[key].label} (colour = temperature)",
    )
    pl.save_figure(figure_3d, FIGS / f"umap_3d_{key}")
    figure_3d.show()

print(f"wrote umap_3d_full.html, umap_3d_pc10.html -> {FIGS}")

## Does the layout organise by anything?

kNN purity (k=15) against two references: `chance` is the purity expected from the class proportions
alone, and `shuffled` is the measured purity after permuting the labels. The input-space columns
matter as much as the embedding columns — if UMAP's purity only matches the input space's, the
embedding is *preserving* structure rather than revealing any.

In [ ]:
structure = gu.label_structure(gene7, spaces, embeddings)
display(structure.round(3))

print("excess over chance (embedding minus chance):")
for label, row in structure.iterrows():
    print(f"  {label:20} input {row['input:full'] - row['chance']:+.3f}"
          f"   umap-2D {row['full:2D'] - row['chance']:+.3f}")

# The detached island in the 2D layouts -- is it a temperature effect or a plate batch effect?
island = embeddings[("full", 2)].to_frame(metadata).query("UMAP1 > 10")
print(f"\ndetached island: n={len(island)} of {len(metadata)}")
for column in ("temperature", "stage", "perturbation_group", "experiment_id"):
    counts = island[column].value_counts().sort_index()
    print(f"  {column:19}" + "  ".join(f"{k}:{v}" for k, v in counts.items()))

## `n_neighbors` sensitivity

At n = 553 the layout is not a fixed object. If a feature survives `n_neighbors` from 5 to 50 it is
probably real; if it appears at one setting only, it is a property of the graph, not the data.

In [ ]:
sweep_layouts, sweep_table = gu.sweep_n_neighbors(spaces, values=(5, 15, 30, 50))
display(sweep_table.pivot(index="n_neighbors", columns="space").round(3))

figure = gu.sweep_figure(sweep_layouts, metadata, spaces, color_by="temperature")
figure.savefig(FIGS / "umap_2d_n_neighbors_sweep.png", dpi=200, bbox_inches="tight")
plt.show()
print("wrote umap_2d_n_neighbors_sweep.png")

## Read-out

**The two input spaces are the same input.** Four PCs carry 90% of the latent variance and seven
carry 99%, so the 70 dimensions that the 10-PC reduction discards hold ~0.5% of it. Every
diagnostic agrees: trustworthiness is 0.986 either way, and kNN purity differs by <0.005 on every
label. Use the 10-PC input — it is cheaper and commensurable with the rest of the GENE7 analysis —
but not because it is better. There is no meaningful choice to make here.

**The layout organises by temperature and stage, not by perturbation.** Temperature purity runs
0.58–0.59 against a chance of 0.25, and stage 0.54 against 0.33. Perturbation group reaches 0.28–0.30
against a chance of 0.25 — barely distinguishable from the label shuffle. A crispant effect is not
visible in an unsupervised nonlinear embedding of this cohort, which is consistent with the rest of
this project needing supervised axes (`gene7_lda_contrasts.ipynb`) to resolve it.

**The detached island is the cold arm, not a batch.** It is ~108 wells, 88 of them 24C, drawn from
four of the six plates and split evenly across all four perturbation groups. Cold rearing slows
development enough that those embryos sit off the main manifold — a temperature effect, not a plate
effect.

**UMAP adds nothing over the linear basis here.** Embedding purity never exceeds input-space purity
on any label, so the nonlinear map preserves structure rather than revealing it, while global
distance ordering degrades (rho 0.81–0.87). The PCA views in `gene7_morphospace.ipynb` are not
missing a nonlinear phenotype. Treat these layouts as a visual companion to that notebook, not as
evidence of anything the PCA missed.

**Caveats.** Every layout is one `random_state`; UMAP has no unique solution and the seed sets the
initialisation. The purity metric is descriptive, not a test — there is no p-value attached, and the
shuffle column is a sanity check on the metric rather than a null model for the phenotype.
`experiment_id` purity (0.30 vs 0.167 chance) partly restates stage, since each plate is a single
timepoint, so it should not be read as an independent batch signal.